In [1]:
#!pip install ultralytics

In [4]:
import torch
from ultralytics import YOLO
import torch.nn as nn

In [8]:
model_v8 = YOLO('yolov8n.pt')
model_v10 = YOLO('yolov10n.pt')
model_v5 = YOLO('yolov5nu.pt')

In [1]:
#model.model.model[-1]

In [2]:
#detect_layer = model.model.model[-1].nc 
#detect_layer = 3

In [9]:
def change_classes(model, new_classes=3):
    
    # 1. Acessar a detection layer
    detect_layer = model.model.model[-1]
    
    # 2. Modificar o atributo nc 
    detect_layer.nc = new_classes

    for i, conv_sequence in enumerate(detect_layer.cv3):
        
        last_conv_idx = None
        for j, layer in enumerate(conv_sequence):
            if isinstance(layer, nn.Conv2d):
                last_conv_idx = j
        
        if last_conv_idx is not None:
            old_conv = conv_sequence[last_conv_idx]
            
            # Criar nova convolução com número correto de classes
            new_conv = nn.Conv2d(
                in_channels=old_conv.in_channels,
                out_channels=new_classes, 
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=old_conv.bias is not None
            )
            
            # SUBSTITUIR a camada antiga pela nova
            conv_sequence[last_conv_idx] = new_conv

    return model

In [10]:
model_v8 = change_classes(model_v8, new_classes=3)
model_v10 = change_classes(model_v10, new_classes=3)
model_v5 = change_classes(model_v5, new_classes=3)